# 25 · Unusual-activity alerts (work package 14)

**Work package 14** of the [applications plan](../../research/implementation_plan_applications.md), section 14,
in its minimal version (`project_direction.md`, 25 November): Dev A for the thresholds, Dev B for the report.

A compromised key often shows as: the account suddenly picks up tokens far outside its own pattern
(**deviation**, from the recommender's ranks), sells what it holds (**liquidation**), and sends the proceeds to an
address it never used (**exit**, from the Transfer logs of WETH, USDC and USDT). An alert fires when all three are
above thresholds set on Dev A to fire at most once per 10,000 active account-days.

**There is no ground truth**: whether an account was compromised is not in the data. What can be checked:

- the alert rate and its stability from Dev A to Dev B;
- **(criterion, stated in the plan)** whether deviation adds to "sold and sent away" in predicting **silence**
  (no trade in the following 7 days): among account-blocks with liquidation and an exit, the silence rate with high
  deviation minus that with low deviation, with a 95% bootstrap interval over days excluding zero (Dev B);
- the strongest alerts of Dev B, prepared for review by the authors with a checklist. They stay local.

**Privacy:** results are in aggregate; no account or address is shown here.

In [1]:
import sys

sys.path.append("../scripts")

import numpy as np
import pandas as pd

import alerts as al
import token_data as td

pd.set_option("display.float_format", "{:,.4f}".format)
WP14 = td.PROJECT_DIR / "wp14"
blocks = pd.read_parquet(WP14 / "blocks.parquet")
account_days = pd.read_csv(WP14 / "account_days.csv", index_col=0)["active_account_days"]
blocks.groupby("part").agg(account_blocks=("user", "size"), accounts=("user", "nunique"), with_exit=("exit_usd", lambda e: (e > 0).mean()),
                           silent=("silent", "mean")).join(account_days)  # fmt: skip

,account_blocks,accounts,with_exit,silent,active_account_days
part,,,,,
dev_a,16160,2489,0.0170,0.0126,78942
dev_b,21240,3659,0.0177,0.0178,97118


Rows are account-blocks with both ranked pick-ups and sales of window tokens, for accounts with at least 5 tokens
of history. The distributions on Dev A:

In [2]:
dev_a = blocks[blocks["part"] == "dev_a"]
dev_a[["deviation", "liquidation", "exit_usd"]].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).T

,count,mean,std,min,50%,75%,90%,95%,99%,max
deviation,"16,160.0000",-0.1840,1.6320,-8.8401,-0.0875,0.8036,1.7085,2.3554,3.5913,7.4536
liquidation,"16,160.0000",0.0507,0.0683,0.0013,0.0282,0.0556,0.1190,0.1667,0.3333,1.0000
exit_usd,"16,160.0000","3,296.7032","172,459.2808",0.0000,0.0000,0.0000,0.0000,0.0000,101.3882,"20,927,581.3809"


## 1. Thresholds on Dev A

The same quantile $q$ for deviation and liquidation, an exit above 0: alerts per 10,000 active account-days.

In [3]:
rows = []
for q in [0.5, 0.75, 0.9, 0.95, 0.99]:
    d_min, l_min = dev_a["deviation"].quantile(q), dev_a["liquidation"].quantile(q)
    n = int(al.fire(dev_a, d_min, l_min).sum())
    rows.append({"q": q, "deviation_min": d_min, "liquidation_min": l_min, "alerts": n, "per_10k_account_days": 1e4 * n / account_days["dev_a"]})
grid = pd.DataFrame(rows)
grid

,q,deviation_min,liquidation_min,alerts,per_10k_account_days
0,0.5000,-0.0875,0.0282,73,9.2473
1,0.7500,0.8036,0.0556,25,3.1669
2,0.9000,1.7085,0.1190,7,0.8867
3,0.9500,2.3554,0.1667,3,0.3800
4,0.9900,3.5913,0.3333,1,0.1267


In [4]:
ok = grid[grid["per_10k_account_days"] <= 1]
chosen = (ok.iloc[0] if len(ok) else grid.iloc[-1])
D_MIN, L_MIN = float(chosen["deviation_min"]), float(chosen["liquidation_min"])
print(f"Chosen: q = {chosen['q']} (deviation >= {D_MIN:.2f}, liquidation >= {L_MIN:.2f}, exit > 0)" + ("" if len(ok) else " -- no q reaches the rate limit; the strictest is used"))

Chosen: q = 0.9 (deviation >= 1.71, liquidation >= 0.12, exit > 0)


## 2. Alert rate on Dev B

In [5]:
blocks["alert"] = al.fire(blocks, D_MIN, L_MIN)
rate = blocks.groupby("part")["alert"].sum().to_frame("alerts").join(account_days)
rate["per_10k_account_days"] = 1e4 * rate["alerts"] / rate["active_account_days"]
rate["accounts_alerted"] = blocks[blocks["alert"]].groupby("part")["user"].nunique()
rate

,alerts,active_account_days,per_10k_account_days,accounts_alerted
part,,,,
dev_a,7,78942,0.8867,6
dev_b,24,97118,2.4712,23


## 3. Does deviation add to "sold and sent away"? (the criterion)

Account-blocks with liquidation at or above its threshold and an exit: silence in the next 7 days, with high
against low deviation. Bootstrap over days (Dev B).

In [6]:
def silence_difference(table: pd.DataFrame, n_boot: int = 5000, seed: int = 0) -> dict:
    """Silence rate (deviation high) minus (deviation low), bootstrap over days."""
    table = table.assign(high=table["deviation"] >= D_MIN)
    days = table["day"].unique()
    by_day = {d: g for d, g in table.groupby("day")}

    def diff(t):
        hi, lo = t[t["high"]], t[~t["high"]]
        return hi["silent"].mean() - lo["silent"].mean() if len(hi) and len(lo) else np.nan

    rng = np.random.default_rng(seed)
    boot = [diff(pd.concat([by_day[d] for d in rng.choice(days, len(days))])) for _ in range(n_boot)]
    low, high = np.nanquantile(boot, [0.025, 0.975])
    return {"rows": len(table), "high_deviation_rows": int(table["high"].sum()), "silent_high": table.loc[table["high"], "silent"].mean(),
            "silent_low": table.loc[~table["high"], "silent"].mean(), "difference": diff(table), "low": low, "high": high, "days": len(days)}  # fmt: skip


selling = blocks[(blocks["liquidation"] >= L_MIN) & (blocks["exit_usd"] > 0)]
checks = pd.DataFrame({part: silence_difference(selling[selling["part"] == part]) for part in ("dev_a", "dev_b")})
checks

,dev_a,dev_b
rows,60.0000,117.0000
high_deviation_rows,7.0000,24.0000
silent_high,0.0000,0.1250
silent_low,0.0755,0.1613
difference,-0.0755,-0.0363
low,-0.1458,-0.1717
high,-0.0185,0.0897
days,17.0000,22.0000


In [7]:
row = checks["dev_b"]
met = row["difference"] > 0 and row["low"] > 0
print(f"Dev B: silence with high deviation {row['silent_high']:.3f} against {row['silent_low']:.3f}, difference {row['difference']:+.3f} "
      f"[{row['low']:+.3f}, {row['high']:+.3f}] -> criterion {'MET' if met else 'NOT met'}")  # fmt: skip

Dev B: silence with high deviation 0.125 against 0.161, difference -0.036 [-0.172, +0.090] -> criterion NOT met


The same without requiring an exit (all account-blocks with liquidation at or above its threshold), as a check of
whether the exit signal matters:

In [8]:
pd.DataFrame({part: silence_difference(blocks[(blocks["liquidation"] >= L_MIN) & (blocks["part"] == part)]) for part in ("dev_a", "dev_b")})

,dev_a,dev_b
rows,"1,619.0000","3,119.0000"
high_deviation_rows,275.0000,569.0000
silent_high,0.0873,0.0984
silent_low,0.0722,0.0906
difference,0.0151,0.0078
low,-0.0079,-0.0265
high,0.0382,0.0448
days,18.0000,22.0000


## 4. The strongest alerts of Dev B, for review

The 50 Dev B alerts with the largest deviation are written to `data/project/wp14/review_dev_b.csv` (local, not
committed) with a checklist for the authors: does the exit address receive from many other accounts, is it
labelled on a public explorer, which tokens were bought. Only counts are shown here.

In [9]:
transfers = None
review = blocks[blocks["alert"] & (blocks["part"] == "dev_b")].nlargest(50, "deviation")
addresses = pd.read_parquet(td.PROJECT_DIR / "users.parquet").set_index("user_id")["address"]
review = review.assign(account=review["user"].map(addresses), silent_7_days=review["silent"],
                       exit_address_receives_from_many="", exit_address_labelled="", reviewer_verdict="")  # fmt: skip
review.drop(columns=["user"]).to_csv(WP14 / "review_dev_b.csv", index=False)
print(f"{len(review)} alerts written for review; {review['silent_7_days'].mean():.0%} of them silent for 7 days; "
      f"median exit ${review['exit_usd'].median():,.0f}")  # fmt: skip

24 alerts written for review; 12% of them silent for 7 days; median exit $88


## 5. What this shows

- **Coverage is thin.** Only about 1.7% of account-blocks with both ranked pick-ups and sales send base tokens to a
  new address within an hour of selling. Most proceeds stay in the account, go back to known addresses, or leave as
  plain ETH, which the Transfer logs do not show. The exit signal is therefore rare, and so are the alerts.
- **The rate does not hold.** With the threshold set on Dev A at q = 0.9 (0.89 alerts per 10,000 active
  account-days), Dev B fires 2.5 per 10,000, about 2.8 times as often. With 7 and 24 alerts, the rate itself is
  uncertain.
- **The criterion is not met.** Among account-blocks that sold and sent proceeds to a new address, high deviation
  does **not** go with more silence afterwards: 12.5% against 16.1% on Dev B, difference −0.036 [−0.172, +0.090].
  Without requiring an exit the difference is +0.008 [−0.027, +0.045]. In this data, "picks up tokens far outside its
  pattern" adds nothing to "sold and sent away" as a sign of an account that is abandoned afterwards.
- **The strongest alerts look like ordinary trading.** Only 12% of the 24 Dev B alerts are silent for the following
  7 days, and the median exit is about \$88. These are small transfers, not a drained account.
- **Limits.** There is no ground truth, and plain ETH transfers are not covered. Dormant accounts, the typical victims
  of a stolen key, are not ranked by WP9 (no trade in the 14 days before t), so they cannot raise a deviation. The
  transfers end on 12 August, so exits in the last day of Dev B are missed. The test period was not downloaded (the
  minimal version), so no test result is reported.

**Verdict:** the recommender's deviation does not make a useful compromise alert here. The application is recorded as
not met.